# Using Real Expression Data

This notebook demonstrates how to use real single-cell RNA sequencing data as input for PointillSim simulations.

## Contents

1. **Why Use Real Data?** - Benefits and considerations
2. **Loading from AnnData** - Using scRNA-seq references
3. **Computing Expression Profiles** - Aggregating to cell type level
4. **Setting Custom Expression** - Manual expression matrix setup
5. **Gene Panel Selection** - Choosing genes for spatial experiments
6. **Transfer Functions** - Modeling platform-specific effects
7. **Complete Example** - End-to-end with real data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    DistanceBasedRule,
    IdentityTransfer,
    AffineNonNegTransfer,
    plot_expression_matrix,
)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

---
## 1. Why Use Real Data?

While synthetic expression profiles are useful for testing, **real scRNA-seq data** provides:

### Benefits

| Aspect | Synthetic | Real Data |
|--------|-----------|----------|
| Expression patterns | Idealized markers | True biological complexity |
| Gene correlations | Random or none | Biological co-expression |
| Dynamic range | User-defined | Realistic distributions |
| Marker specificity | Perfect | Leaky, overlapping |

### Common Sources

- **Published atlases**: Human Cell Atlas, Tabula Sapiens, etc.
- **GEO/ArrayExpress**: Public scRNA-seq datasets
- **Your own data**: Internal references

### Workflow Overview

1. Load scRNA-seq AnnData with cell type annotations
2. Compute mean expression per cell type
3. Select gene panel (if not using all genes)
4. Apply transfer function for platform effects
5. Generate spatial simulations

---
## 2. Loading from AnnData

Most scRNA-seq data is distributed as AnnData objects. Let's see how to use them.

### Creating a Mock Dataset

For this tutorial, we'll create a mock scRNA-seq dataset. In practice, you would load your own data.

In [ ]:
# Create a mock scRNA-seq dataset
try:
    import anndata as ad
    import scipy.sparse as sp
    
    # Simulate 2000 cells, 100 genes, 4 cell types
    n_cells = 2000
    n_genes = 100
    n_types = 4
    
    cell_type_names = ['T cell', 'B cell', 'Monocyte', 'NK cell']
    gene_names = [f'Gene_{i:03d}' for i in range(n_genes)]
    
    # Assign cell types
    cell_types = np.random.choice(cell_type_names, size=n_cells, p=[0.4, 0.25, 0.25, 0.1])
    
    # Create expression matrix with type-specific patterns
    # Each cell type has ~10 marker genes
    X = np.random.poisson(2, size=(n_cells, n_genes)).astype(float)
    
    for i, ct in enumerate(cell_type_names):
        mask = cell_types == ct
        # Marker genes for this type: genes i*25 to i*25+10
        marker_start = i * 25
        X[mask, marker_start:marker_start+10] += np.random.poisson(15, size=(mask.sum(), 10))
    
    # Create AnnData
    adata = ad.AnnData(
        X=sp.csr_matrix(X),
        obs=pd.DataFrame({'cell_type': cell_types}, index=[f'cell_{i}' for i in range(n_cells)]),
        var=pd.DataFrame(index=gene_names),
    )
    
    print(f"Created mock scRNA-seq dataset:")
    print(f"  Cells: {adata.n_obs}")
    print(f"  Genes: {adata.n_vars}")
    print(f"  Cell types: {adata.obs['cell_type'].nunique()}")
    print(f"\nCell type distribution:")
    print(adata.obs['cell_type'].value_counts())
    
    ANNDATA_AVAILABLE = True
    
except ImportError:
    print("Install anndata to use this feature: pip install anndata")
    print("\nContinuing with synthetic data...")
    ANNDATA_AVAILABLE = False

---
## 3. Computing Expression Profiles

PointillSim needs a **genes × cell_types** expression matrix. We compute this by averaging expression within each cell type.

In [ ]:
def compute_expression_profiles(adata, cell_type_col='cell_type', use_raw=False):
    """
    Compute mean expression per cell type from an AnnData object.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data matrix with cell type annotations
    cell_type_col : str
        Column in adata.obs containing cell type labels
    use_raw : bool
        If True, use adata.raw (unnormalized counts)
        
    Returns
    -------
    expression_df : pd.DataFrame
        Genes × Cell Types mean expression matrix
    """
    import scipy.sparse as sp
    
    # Get expression matrix
    if use_raw and adata.raw is not None:
        X = adata.raw.X
        gene_names = adata.raw.var_names
    else:
        X = adata.X
        gene_names = adata.var_names
    
    # Convert to dense if sparse
    if sp.issparse(X):
        X = X.toarray()
    
    # Get cell types
    cell_types = adata.obs[cell_type_col].values
    unique_types = sorted(adata.obs[cell_type_col].unique())
    
    # Compute mean per type
    mean_expr = np.zeros((len(gene_names), len(unique_types)))
    for i, ct in enumerate(unique_types):
        mask = cell_types == ct
        mean_expr[:, i] = X[mask].mean(axis=0)
    
    # Create DataFrame
    expression_df = pd.DataFrame(
        mean_expr,
        index=gene_names,
        columns=unique_types
    )
    
    return expression_df

In [ ]:
if ANNDATA_AVAILABLE:
    # Compute expression profiles
    expr_df = compute_expression_profiles(adata, cell_type_col='cell_type')
    
    print(f"Expression matrix shape: {expr_df.shape}")
    print(f"  Genes: {expr_df.shape[0]}")
    print(f"  Cell types: {list(expr_df.columns)}")
    
    # Show top expressed genes per type
    print("\nTop 5 genes per cell type:")
    for ct in expr_df.columns:
        top_genes = expr_df[ct].nlargest(5).index.tolist()
        print(f"  {ct}: {', '.join(top_genes)}")

In [ ]:
if ANNDATA_AVAILABLE:
    # Visualize the expression matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    
    # Full matrix (subset of genes for visibility)
    ax = axes[0]
    top_genes = expr_df.max(axis=1).nlargest(40).index
    im = ax.imshow(np.log1p(expr_df.loc[top_genes].values), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(expr_df.columns)))
    ax.set_xticklabels(expr_df.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(top_genes)))
    ax.set_yticklabels(top_genes, fontsize=8)
    ax.set_title('Top 40 Expressed Genes\n(log scale)')
    plt.colorbar(im, ax=ax, label='log(expr+1)')
    
    # Marker gene heatmap
    ax = axes[1]
    # Get top 5 markers per type
    marker_genes = []
    for ct in expr_df.columns:
        # Marker score: expression in this type / max in other types
        others = [c for c in expr_df.columns if c != ct]
        marker_score = expr_df[ct] / (expr_df[others].max(axis=1) + 0.1)
        marker_genes.extend(marker_score.nlargest(5).index.tolist())
    marker_genes = list(dict.fromkeys(marker_genes))  # Remove duplicates, keep order
    
    im = ax.imshow(np.log1p(expr_df.loc[marker_genes].values), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(expr_df.columns)))
    ax.set_xticklabels(expr_df.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(marker_genes)))
    ax.set_yticklabels(marker_genes, fontsize=8)
    ax.set_title('Top 5 Marker Genes per Type\n(log scale)')
    plt.colorbar(im, ax=ax, label='log(expr+1)')
    
    plt.tight_layout()
    plt.show()

---
## 4. Setting Custom Expression

Now let's create a `TissueCellTypes` object with our real expression data.

In [ ]:
def tissue_from_expression_df(expr_df):
    """
    Create a TissueCellTypes object from an expression DataFrame.
    
    Parameters
    ----------
    expr_df : pd.DataFrame
        Genes × Cell Types expression matrix
        
    Returns
    -------
    tissue : TissueCellTypes
        Configured tissue object
    """
    tissue = TissueCellTypes()
    
    # Set expression matrix
    tissue.gene_expression_by_type = expr_df.values.copy()
    
    # Set gene names
    tissue._gene_names = list(expr_df.index)
    
    # Set cell type names
    tissue._cell_type_names = list(expr_df.columns)
    
    return tissue

In [ ]:
if ANNDATA_AVAILABLE:
    # Create tissue from real data
    tissue_real = tissue_from_expression_df(expr_df)
    
    print(f"Created TissueCellTypes from real data:")
    print(f"  Genes: {tissue_real.n_genes}")
    print(f"  Cell types: {tissue_real.n_cell_types}")
    print(f"  Gene names: {tissue_real.gene_names[:5]}...")
    print(f"  Cell type names: {tissue_real.cell_type_names}")
else:
    # Use synthetic data
    tissue_real = TissueCellTypes()
    tissue_real.generate_types_and_markers(
        n_genes=50,
        n_cell_types=4,
        expected_level=10.0,
        concentration=0.85,
    )
    print("Using synthetic tissue (anndata not available)")

---
## 5. Gene Panel Selection

Spatial transcriptomics platforms typically use a **targeted gene panel** (50-500 genes), not the full transcriptome.

### Selection Strategies

| Strategy | Description | Use Case |
|----------|-------------|----------|
| Marker genes | Highest expression ratio between types | Cell type classification |
| Highly variable | Genes with high variance across cells | Discovery/clustering |
| Pathway-specific | Genes in specific pathways | Biological questions |
| Commercial panel | Pre-designed panels (Xenium, MERFISH) | Platform compatibility |

In [ ]:
def select_marker_genes(expr_df, n_markers_per_type=10, min_expression=1.0):
    """
    Select marker genes based on specificity to each cell type.
    
    Parameters
    ----------
    expr_df : pd.DataFrame
        Genes × Cell Types expression matrix
    n_markers_per_type : int
        Number of markers to select per cell type
    min_expression : float
        Minimum expression level to consider
        
    Returns
    -------
    selected_genes : list
        List of selected gene names
    """
    selected = []
    
    for ct in expr_df.columns:
        # Calculate marker score: expression in this type vs max in others
        others = [c for c in expr_df.columns if c != ct]
        
        # Avoid division by zero
        max_others = expr_df[others].max(axis=1) + 0.1
        marker_score = expr_df[ct] / max_others
        
        # Filter by minimum expression
        marker_score = marker_score[expr_df[ct] >= min_expression]
        
        # Select top markers
        top_markers = marker_score.nlargest(n_markers_per_type).index.tolist()
        selected.extend(top_markers)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_selected = []
    for gene in selected:
        if gene not in seen:
            seen.add(gene)
            unique_selected.append(gene)
    
    return unique_selected

In [ ]:
if ANNDATA_AVAILABLE:
    # Select marker panel
    panel_genes = select_marker_genes(expr_df, n_markers_per_type=10, min_expression=1.0)
    
    print(f"Selected {len(panel_genes)} marker genes")
    print(f"\nGenes: {panel_genes[:10]}...")
    
    # Create subset tissue
    expr_panel = expr_df.loc[panel_genes]
    tissue_panel = tissue_from_expression_df(expr_panel)
    
    print(f"\nPanel tissue: {tissue_panel.n_genes} genes, {tissue_panel.n_cell_types} types")

In [ ]:
if ANNDATA_AVAILABLE:
    # Visualize panel expression
    fig, ax = plt.subplots(figsize=(10, 12))
    
    im = ax.imshow(np.log1p(expr_panel.values), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(expr_panel.columns)))
    ax.set_xticklabels(expr_panel.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(panel_genes)))
    ax.set_yticklabels(panel_genes, fontsize=8)
    ax.set_xlabel('Cell Type')
    ax.set_ylabel('Gene')
    ax.set_title(f'Selected Gene Panel ({len(panel_genes)} genes)\n(log scale)')
    plt.colorbar(im, ax=ax, label='log(expr+1)')
    
    plt.tight_layout()
    plt.show()

---
## 6. Transfer Functions

Spatial transcriptomics platforms have different detection characteristics than scRNA-seq. The **transfer function** models these differences.

### Common Effects

| Effect | Description | Modeling |
|--------|-------------|----------|
| Gene efficiency | Probe/detection efficiency varies by gene | Gene-specific scaling |
| Background | Ambient RNA, autofluorescence | Additive offset |
| Saturation | High expression may plateau | Non-linear transform |
| Dropout | Low expression may not be detected | Zero-inflation |

In [ ]:
# Identity transfer (no transformation)
identity_tf = IdentityTransfer()

# Affine transfer (linear with gene-specific effects)
affine_tf = AffineNonNegTransfer(
    scales=1.0,       # Mean scale factor
    scales_std=0.5,   # Variation across genes (50% CV)
    offsets=0.0,      # Mean offset (background)
    offsets_std=0.1,  # Variation in background
)

print("Transfer functions:")
print(f"  Identity: no transformation")
print(f"  Affine: scale~N(1.0, 0.5), offset~N(0.0, 0.1)")

In [ ]:
if ANNDATA_AVAILABLE:
    # Compare original vs transformed expression
    original = tissue_panel.gene_expression_by_type.copy()
    transformed = affine_tf.transform(original)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 8))
    
    # Original
    ax = axes[0]
    im = ax.imshow(np.log1p(original), aspect='auto', cmap='viridis')
    ax.set_title('Original Expression\n(scRNA-seq reference)')
    ax.set_xlabel('Cell Type')
    ax.set_ylabel('Gene')
    plt.colorbar(im, ax=ax)
    
    # Transformed
    ax = axes[1]
    im = ax.imshow(np.log1p(transformed), aspect='auto', cmap='viridis')
    ax.set_title('Transformed Expression\n(simulated spatial)')
    ax.set_xlabel('Cell Type')
    ax.set_ylabel('Gene')
    plt.colorbar(im, ax=ax)
    
    # Gene-specific scales
    ax = axes[2]
    ax.barh(range(len(affine_tf.scales)), affine_tf.scales)
    ax.set_xlabel('Scale Factor')
    ax.set_ylabel('Gene Index')
    ax.set_title('Gene-Specific Detection\nEfficiency (scale)')
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

---
## 7. Complete Example

Let's put it all together: load data, select genes, configure tissue, and generate FOVs.

In [ ]:
# Use panel tissue if available, otherwise synthetic
if ANNDATA_AVAILABLE:
    tissue = tissue_panel
    cell_type_names = tissue.cell_type_names
else:
    tissue = tissue_real
    cell_type_names = [f'Type {i}' for i in range(tissue.n_cell_types)]

n_cell_types = tissue.n_cell_types
print(f"Using tissue with {tissue.n_genes} genes and {n_cell_types} cell types")
print(f"Cell types: {cell_type_names}")

In [ ]:
# Configure cell properties
cell_props = CellTypesProperties(
    n_cell_types=n_cell_types,
    sizes=[10, 12, 14, 9],  # Different sizes per type
    anisotropy=[0.9, 0.85, 0.8, 0.95],  # Shape variation
    relative_rna_concentration=[1.0, 1.2, 1.5, 0.8],  # RNA content
)

In [ ]:
# Create FOV distribution with realistic structure
frame_size = 600

# Background: mix of two main types
background = lambda: FrameWideElement(
    frame_size=frame_size,
    tipical_cell_spacing=15,
    rules=MixOfNCellTypesRule(
        n_cell_types=n_cell_types,
        list_N=[0, 1],  # T cells and B cells
        proportions=[0.6, 0.4]
    )
)

# Germinal center-like structure (B cell rich)
gc_structure = lambda: VacuolatedStructure(
    frame_size=frame_size,
    scale=80,
    hole_scale_factor=0.4,
    tipical_cell_spacing=10,
    rules=DistanceBasedRule(
        n_cell_types=n_cell_types,
        inner_type=1,  # B cells in center
        outer_type=0,  # T cells at periphery
        transition_width=20
    )
)

# Monocyte cluster
from pointillsim import HistologicalElement, SingleTypeRule

monocyte_cluster = lambda: HistologicalElement(
    frame_size=frame_size,
    scale=40,
    tipical_cell_spacing=8,
    rules=SingleTypeRule(n_cell_types=n_cell_types, cell_type_ix=2)
)

fov_dist = FOVDistribution(
    frame_size=frame_size,
    background_element=background,
    other_elements=[gc_structure, monocyte_cluster],
    elements_frequency=[0.5, 0.4],
    attempts_at_elements=[2, 3],
)

In [ ]:
# Generate FOV
np.random.seed(42)
fov = fov_dist.generate_fov()
cell_props.apply(fov)

# Generate observations with transfer function
hybiss = HybISS_Setup(
    tissue=tissue,
    genes_sensitivities=1.0,
    genes_sensitivities_variation=0.3,
    transfer_function=AffineNonNegTransfer(
        scales=1.0, scales_std=0.4,
        offsets=0.0, offsets_std=0.05
    )
)
hybiss.observe_dots(fov)

dots_df = hybiss.make_pandas_df()
cells_df = fov.make_pandas_df()

print(f"Generated FOV:")
print(f"  Cells: {fov.n_cells}")
print(f"  Dots: {len(dots_df)}")
print(f"\nCell type distribution:")
for i, name in enumerate(cell_type_names):
    count = (fov.class_instance == i).sum()
    print(f"  {name}: {count} ({100*count/fov.n_cells:.1f}%)")

In [ ]:
# Visualize the complete simulation
from matplotlib.patches import Ellipse, Patch
from matplotlib.collections import PatchCollection

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# A: Cell positions by type
ax = axes[0, 0]
colors = plt.cm.Set2(np.linspace(0, 1, n_cell_types))
for i, (name, color) in enumerate(zip(cell_type_names, colors)):
    mask = fov.class_instance == i
    ax.scatter(
        fov.cell_centroids[mask, 0],
        fov.cell_centroids[mask, 1],
        c=[color], s=20, alpha=0.7, label=name
    )
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('A. Cell Types')
ax.legend(loc='upper right')

# B: Cell morphology (zoomed)
ax = axes[0, 1]
x_min, x_max = 150, 450
y_min, y_max = 150, 450
mask = (
    (fov.cell_centroids[:, 0] >= x_min) & (fov.cell_centroids[:, 0] <= x_max) &
    (fov.cell_centroids[:, 1] >= y_min) & (fov.cell_centroids[:, 1] <= y_max)
)
ellipses = []
cell_colors = []
for i in np.where(mask)[0]:
    ellipse = Ellipse(
        xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
        width=2 * fov.cell_major_axis[i],
        height=2 * fov.cell_minor_axis[i],
        angle=np.degrees(fov.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    cell_colors.append(fov.cell_colors[i])
collection = PatchCollection(ellipses, alpha=0.6)
collection.set_facecolors(cell_colors)
collection.set_edgecolors('black')
collection.set_linewidths(0.3)
ax.add_collection(collection)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('B. Cell Morphology (zoomed)')

# C: Transcript dots by gene (top 5 genes)
ax = axes[1, 0]
top_genes = dots_df['gene'].value_counts().head(5).index
gene_colors = plt.cm.Set1(np.linspace(0, 1, 5))
for gene, color in zip(top_genes, gene_colors):
    gene_mask = dots_df['gene'] == gene
    ax.scatter(
        dots_df.loc[gene_mask, 'x'],
        dots_df.loc[gene_mask, 'y'],
        c=[color], s=3, alpha=0.5, label=gene
    )
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'C. Transcript Dots (top 5 genes)')
ax.legend(loc='upper right', markerscale=3)

# D: Gene expression by cell type
ax = axes[1, 1]
# Aggregate dots to cell × gene counts
counts_per_cell = dots_df.groupby(['cell', 'gene']).size().unstack(fill_value=0)
mean_per_type = np.zeros((n_cell_types, min(10, len(counts_per_cell.columns))))
genes_subset = counts_per_cell.columns[:10]
for ct in range(n_cell_types):
    ct_cells = np.where(fov.class_instance == ct)[0]
    ct_cells = [c for c in ct_cells if c in counts_per_cell.index]
    if len(ct_cells) > 0:
        mean_per_type[ct] = counts_per_cell.loc[ct_cells, genes_subset].mean().values

im = ax.imshow(mean_per_type, aspect='auto', cmap='viridis')
ax.set_yticks(range(n_cell_types))
ax.set_yticklabels(cell_type_names)
ax.set_xticks(range(len(genes_subset)))
ax.set_xticklabels(genes_subset, rotation=45, ha='right', fontsize=8)
ax.set_title('D. Mean Counts per Cell Type')
ax.set_xlabel('Gene')
ax.set_ylabel('Cell Type')
plt.colorbar(im, ax=ax, label='Mean counts')

plt.tight_layout()
plt.show()

In [ ]:
# Export to AnnData for downstream analysis
try:
    adata_out = fov.to_anndata(tissue)
    print(f"Exported to AnnData:")
    print(f"  Cells: {adata_out.n_obs}")
    print(f"  Genes: {adata_out.n_vars}")
    print(f"  obs columns: {list(adata_out.obs.columns)}")
    print(f"  obsm keys: {list(adata_out.obsm.keys())}")
except ImportError:
    print("Install anndata for AnnData export")

---
## Summary

This notebook covered using real scRNA-seq data with PointillSim:

| Step | Function | Purpose |
|------|----------|--------|
| Load AnnData | `ad.read_h5ad()` | Load scRNA-seq reference |
| Compute profiles | `compute_expression_profiles()` | Mean expression per type |
| Select genes | `select_marker_genes()` | Create spatial gene panel |
| Create tissue | `tissue_from_expression_df()` | Configure PointillSim |
| Apply transfer | `AffineNonNegTransfer` | Model platform effects |

### Key Points

1. **Real expression data** provides biologically realistic patterns
2. **Gene selection** is critical for spatial experiments
3. **Transfer functions** model platform-specific effects
4. **Export to AnnData** enables standard analysis pipelines

### Next Steps

- **05_cell_type_rules.ipynb**: Deep dive into spatial cell type rules
- **06_tissue_simulations.ipynb**: Simulating specific real tissues